<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## Logical Plan Nodes 🧩

These classes represent the relational operations in our query plan.

In [0]:
#| echo: false
#| output: asis
show_doc(Join)

---

### Join

```python

def Join(
    left:LogicalPlan, right:LogicalPlan, left_on:List, right_on:List, how:str='inner'
)->None:


```

*Represents an inner join between two plan trees (first two-child node).*

In [0]:
#| echo: false
#| output: asis
show_doc(Distinct)

---

### Distinct

```python

def Distinct(
    input:LogicalPlan, subset:Optional=None
)->None:


```

*Represents removing duplicate rows (optionally on a subset of columns).*

In [0]:
#| echo: false
#| output: asis
show_doc(Limit)

---

### Limit

```python

def Limit(
    input:LogicalPlan, n:int
)->None:


```

*Represents taking only the first N rows.*

In [0]:
#| echo: false
#| output: asis
show_doc(Sort)

---

### Sort

```python

def Sort(
    input:LogicalPlan, by:List, descending:List
)->None:


```

*Represents sorting rows by one or more keys.*

In [0]:
#| echo: false
#| output: asis
show_doc(Aggregate)

---

### Aggregate

```python

def Aggregate(
    input:LogicalPlan, group_by:List, aggs:List
)->None:


```

*Represents grouping and aggregating data.*

In [0]:
#| echo: false
#| output: asis
show_doc(Project)

---

### Project

```python

def Project(
    input:LogicalPlan, exprs:List
)->None:


```

*Represents selecting or computing columns.*

In [0]:
#| echo: false
#| output: asis
show_doc(Filter)

---

### Filter

```python

def Filter(
    input:LogicalPlan, predicate:Expr
)->None:


```

*Represents filtering rows based on a predicate.*

In [0]:
#| echo: false
#| output: asis
show_doc(Scan)

---

### Scan

```python

def Scan(
    table:Table
)->None:


```

*Represents reading data from a source (e.g., a PyArrow Table).*

In [0]:
#| echo: false
#| output: asis
show_doc(LogicalPlan)

---

### LogicalPlan

```python

def LogicalPlan(
    args:VAR_POSITIONAL, kwargs:VAR_KEYWORD
):


```

*Base class for all logical plan nodes.*

## The `LazyFrame` API 🚀

This is the main entry point for users to build queries.

In [0]:
#| echo: false
#| output: asis
show_doc(LazyGroupBy)

---

### LazyGroupBy

```python

def LazyGroupBy(
    df, by
):


```

*Intermediate object for grouping operations.*

In [0]:
#| echo: false
#| output: asis
show_doc(LazyFrame)

---

### LazyFrame

```python

def LazyFrame(
    plan
):


```

*A lazy DataFrame that builds a logical plan.*

## Tests 🧪

Let's verify that our `LazyFrame` builds the correct `LogicalPlan`.

In [ ]:
import pyarrow as pa
from mxframe.lazy_expr import col, lit

# Dummy data
table = pa.table({'a': [1, 2, 3], 'b': [4, 5, 6]})
lf = LazyFrame(Scan(table))

# Test select
lf_sel = lf.select(col('a') + lit(1), 'b')
assert isinstance(lf_sel.plan, Project)
assert len(lf_sel.plan.exprs) == 2
assert lf_sel.plan.exprs[1].op == 'col'

# Test filter
lf_fil = lf.filter(col('a') > lit(1))
assert isinstance(lf_fil.plan, Filter)
assert lf_fil.plan.predicate.op == 'gt'

# Test groupby
lf_agg = lf.groupby('a').agg(col('b').sum())
assert isinstance(lf_agg.plan, Aggregate)
assert len(lf_agg.plan.group_by) == 1
assert len(lf_agg.plan.aggs) == 1
assert lf_agg.plan.aggs[0].op == 'sum'

print("All tests passed! 🎉")

# ── Phase 4 Tests: Sort, Limit, Distinct ──────────────────────────────────────
from mxframe.lazy_frame import Sort, Limit, Distinct

# ── Test: .sort() builds Sort node ──
lf_sort = LazyFrame(Scan(table)).sort('a', 'b')
assert isinstance(lf_sort.plan, Sort), f'Expected Sort, got {type(lf_sort.plan)}'
assert len(lf_sort.plan.by) == 2
assert lf_sort.plan.descending == [False, False]
print('✅ .sort() builds Sort node')

# ── Test: .sort() with descending ──
lf_sort_desc = LazyFrame(Scan(table)).sort('a', descending=True)
assert lf_sort_desc.plan.descending == [True]
print('✅ .sort() descending flag works')

# ── Test: .limit() builds Limit node ──
lf_limit = LazyFrame(Scan(table)).limit(10)
assert isinstance(lf_limit.plan, Limit)
assert lf_limit.plan.n == 10
print('✅ .limit() builds Limit node')

# ── Test: .distinct() builds Distinct node ──
lf_dist = LazyFrame(Scan(table)).distinct('a')
assert isinstance(lf_dist.plan, Distinct)
assert lf_dist.plan.subset == ['a']
print('✅ .distinct() builds Distinct node')

# ── Test: .distinct() with no args uses None subset ──
lf_dist_all = LazyFrame(Scan(table)).distinct()
assert lf_dist_all.plan.subset is None
print('✅ .distinct() with no args works')

# ── Test: Chaining sort + limit ──
lf_chain = LazyFrame(Scan(table)).sort('a').limit(5)
assert isinstance(lf_chain.plan, Limit)
assert isinstance(lf_chain.plan.input, Sort)
print('✅ sort + limit chaining works')

# ── Test: Signatures are hashable ──
sig1 = lf_sort.plan.signature()
sig2 = lf_limit.plan.signature()
sig3 = lf_dist.plan.signature()
s = {sig1, sig2, sig3}
assert len(s) == 3
print('✅ Sort/Limit/Distinct signatures hashable')

print('\nAll Phase 4 plan node tests passed! 🎉')


# --- Join tests ---
left_t = pa.table({"key": [1, 2, 3], "val_l": [10, 20, 30]})
right_t = pa.table({"key": [2, 3, 4], "val_r": [200, 300, 400]})
lf = LazyFrame(left_t)
rf = LazyFrame(right_t)

# on= shorthand
jnode = lf.join(rf, on="key").plan
assert isinstance(jnode, Join)
assert jnode.left_on == ["key"] and jnode.right_on == ["key"]
assert jnode.how == "inner"

# left_on/right_on
jnode2 = lf.join(rf, left_on="key", right_on="key").plan
assert isinstance(jnode2, Join)

# multi-key
jnode3 = lf.join(rf, on=["key"]).plan
assert jnode3.left_on == ["key"]

# signature is hashable
sig = jnode.signature()
assert isinstance(sig, tuple)

# two children
assert hasattr(jnode, 'left') and hasattr(jnode, 'right')
assert isinstance(jnode.left, Scan) and isinstance(jnode.right, Scan)

# chained joins: A.join(B).join(C)
third_t = pa.table({"key": [2, 3], "val_t": [2000, 3000]})
tf = LazyFrame(third_t)
chained = lf.join(rf, on="key").join(tf, on="key").plan
assert isinstance(chained, Join)
assert isinstance(chained.left, Join)  # inner join is left child

print("✓ all join plan tests passed")

All tests passed! 🎉
✅ .sort() builds Sort node
✅ .sort() descending flag works
✅ .limit() builds Limit node
✅ .distinct() builds Distinct node
✅ .distinct() with no args works
✅ sort + limit chaining works
✅ Sort/Limit/Distinct signatures hashable

All Phase 4 plan node tests passed! 🎉
✓ all join plan tests passed
